# Cocoyam — Data Integration and Statistical Analysis

This notebook processes LC-MS/MS metabolomics data for cocoyam (*Colocasia esculenta*) samples from batch `b3_cocoyamonly`. It follows the same pipeline as the cassava notebook: consolidating MS2 annotations, running differential abundance statistics between control and fermented conditions, and preparing compound structures for BioTransformer.

- **Section 1** — MS2 annotation processing (SIRIUS + GNPS + Suspect)
- **Section 2** — Quantitative analysis (normalization, imputation, FDR-corrected statistics)
- **Section 3** — BioTransformer prep (InChIKey → SMILES)

**Inputs:** SIRIUS/CANOPUS predictions, GNPS spectral library and suspect list matches, MZmine quantitative feature table.

**Outputs:** `b3_cocoyamonly_ms2_annotations.csv`, `b3_cocoyamonly_feature_list_imputed_log2.csv`, `b3_cocoyamonly_fdr_cocoyam_vs_fermented_cocoyam.csv`, BioTransformer input file.

In [1]:
import pandas as pd
import numpy as np
import re
import os
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ── Configuration ──────────────────────────────────────────────────────────
BATCH_ID   = "b3_cocoyamonly"
DATA_PATH  = f"../data/processed_data/{BATCH_ID}_biot"  # New biotransformer SIRIUS results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Batch: {BATCH_ID}")
print(f"Data path: {DATA_PATH}")
print(f"Random seed: {RANDOM_SEED}")

Batch: b3_cocoyamonly
Data path: ../data/processed_data/b3_cocoyamonly_biot
Random seed: 42


---
## Section 1 — MS2 Annotation Processing

Annotations from three sources are consolidated per feature: SIRIUS structure predictions (filtered at confidence ≥ 0.64), CANOPUS chemical classifications (NPC and ClassyFire taxonomies), and GNPS spectral library matches. A GNPS suspect list provides additional tentative identifications. Where both GNPS and SIRIUS annotate the same feature, InChIKey agreement is checked. Feature m/z and retention time come from the MZmine quantitative feature table.

### 1.1 Load SIRIUS Structure Identifications

In [2]:
sirius_structure_annotations = pd.read_csv(
    f"{DATA_PATH}/structure_identifications.tsv", sep="\t"
)
print(f"SIRIUS structure annotations shape: {sirius_structure_annotations.shape}")
print("Columns:", sirius_structure_annotations.columns.tolist())
sirius_structure_annotations.head()

SIRIUS structure annotations shape: (984, 27)
Columns: ['structurePerIdRank', 'formulaRank', 'ConfidenceScoreExact', 'ConfidenceScoreApproximate', 'CSI:FingerIDScore', 'ZodiacScore', 'SiriusScoreNormalized', 'SiriusScore', 'molecularFormula', 'adduct', 'precursorFormula', 'InChIkey2D', 'InChI', 'name', 'smiles', 'xlogp', 'pubchemids', 'links', 'dbflags', 'ionMass', 'retentionTimeInSeconds', 'retentionTimeInMinutes', 'formulaId', 'alignedFeatureId', 'compoundId', 'mappingFeatureId', 'overallFeatureQuality']


,structurePerIdRank,formulaRank,ConfidenceScoreExact,ConfidenceScoreApproximate,CSI:FingerIDScore,ZodiacScore,SiriusScoreNormalized,SiriusScore,molecularFormula,adduct,...,links,dbflags,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,compoundId,mappingFeatureId,overallFeatureQuality
0,1,1,0.125,0.125,-85.331,NaN,0.893,14.245,C6H6FO3PS,[M + K]+,...,PUBCHEMANNOTATIONBIO:(null);PUBCHEM:(49799138)...,16777282,246.937,36,0.593,816964643019809945,816964165406012804,816964165376652675,6,NaN
1,1,1,0.180,0.180,-197.594,NaN,0.950,31.218,C16H32N2O10,[M + H]+,...,PUBCHEM:(101202401),2,413.211,36,0.606,816964648262691053,816964165577979273,816964165573784968,7,NaN
2,1,1,0.157,0.157,-82.004,NaN,0.995,32.190,C6H11O5P,[M + K]+,...,NORMAN:(NS00033255);PUBCHEMANNOTATIONSAFETYAND...,137506324482,232.998,37,0.620,816964725454676182,816964165703808398,816964165703808397,26,NaN
3,1,1,0.007,0.007,-202.738,NaN,0.987,8.837,C11H24N2O5,[M + H]+,...,PUBCHEM:(113426778),2,265.175,37,0.621,816964648371742993,816964165812860307,816964165812860306,27,NaN
4,1,1,0.235,0.235,-85.568,NaN,0.986,30.643,C7H13O5P,[M + K]+,...,NORMAN:(NS00057775);PUBCHEM:(20707859 103972);...,137439215618,247.014,37,0.623,816964672669350467,816964166160987554,816964166160987553,40,NaN


In [3]:
confidence_threshold = 0.64

# Keep only rank-1 structure per feature (best hit)
sirius_best = sirius_structure_annotations[
    sirius_structure_annotations['structurePerIdRank'] == 1
].copy()

# Flag high-confidence annotations
sirius_best['high_confidence_sirius'] = (
    sirius_best['ConfidenceScoreApproximate'] >= confidence_threshold
)

# Select columns that feed into the final annotations
sirius_cols_keep = [
    'mappingFeatureId',
    'ConfidenceScoreApproximate',
    'InChIkey2D', 'InChI', 'name', 'smiles', 'dbflags', 'links',
    'high_confidence_sirius'
]
# Keep only existing cols (safety)
sirius_cols_keep = [c for c in sirius_cols_keep if c in sirius_best.columns]
sirius_struct_clean = sirius_best[sirius_cols_keep].copy()

print(f"SIRIUS rank-1 features: {len(sirius_struct_clean)}")
print(f"High-confidence (>={confidence_threshold}): {sirius_struct_clean['high_confidence_sirius'].sum()}")
sirius_struct_clean.head()

SIRIUS rank-1 features: 984
High-confidence (>=0.64): 125


,mappingFeatureId,ConfidenceScoreApproximate,InChIkey2D,InChI,name,smiles,dbflags,links,high_confidence_sirius
0,6,0.125,RDAVYJKCJHOMSD,"InChI=1S/C6H6FO3PS/c7-4-1-2-6(12)5(3-4)11(8,9)...",5-Fluoro-2-sulfanyl-phenylphosphonic acid,C1=CC(=C(C=C1F)P(=O)(O)O)S,16777282,PUBCHEMANNOTATIONBIO:(null);PUBCHEM:(49799138)...,False
1,7,0.180,TVBWHGVMDKQHJJ,InChI=1S/C16H32N2O10/c19-5-7-9(21)11(23)13(25)...,"N,N'-Bis(alpha-D-mannopyranosyl)-1,4-butanedia...",C(CCNC1C(C(C(C(O1)CO)O)O)O)CNC2C(C(C(C(O2)CO)O...,2,PUBCHEM:(101202401),False
2,26,0.157,HRTGCDRCJQKACR,"InChI=1S/C6H11O5P/c1-5(6(7)9-2)12(8,10-3)11-4/...",Methyl 2-(dimethoxyphosphinyl)acrylate,COC(=O)C(=C)P(=O)(OC)OC,137506324482,NORMAN:(NS00033255);PUBCHEMANNOTATIONSAFETYAND...,False
3,27,0.007,BIOQXNMMINDLQA,InChI=1S/C11H24N2O5/c1-15-4-5-16-6-7-17-8-9-18...,N-(2-aminoethyl)-2-[2-[2-(2-methoxyethoxy)etho...,COCCOCCOCCOCC(=O)NCCN,2,PUBCHEM:(113426778),False
4,40,0.235,STCBUTTVBMCYJL,"InChI=1S/C7H13O5P/c1-13(12,4-2-6(8)9)5-3-7(10)...","Propanoic acid, 3,3'-(methylphosphinylidene)bis-",CP(=O)(CCC(=O)O)CCC(=O)O,137439215618,NORMAN:(NS00057775);PUBCHEM:(20707859 103972);...,False


### 1.2 Load CANOPUS Formula Annotations

In [4]:
canopus_annotations = pd.read_csv(
    f"{DATA_PATH}/canopus_structure_summary.tsv", sep="\t"
)
print(f"CANOPUS annotations shape: {canopus_annotations.shape}")
canopus_annotations.head()

CANOPUS annotations shape: (984, 29)


,formulaRank,molecularFormula,adduct,precursorFormula,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,...,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,compoundId,mappingFeatureId,overallFeatureQuality
0,1,C6H6FO3PS,[M + K]+,C6H6FKO3PS+,Fatty acids,0.075,Pseudoalkaloids (transamidation),0.061,Halogenated fatty acids,0.030,...,0.815,Organic compounds; Organic acids and derivativ...,246.937,36,0.593,816964643019809945,816964165406012804,816964165376652675,6,NaN
1,1,C16H32N2O10,[M + H]+,C16H33N2O10+,Carbohydrates,0.990,Aminosugars and aminoglycosides,0.765,Amino cyclitols,0.730,...,0.528,Organic compounds; Organoheterocyclic compound...,413.211,36,0.606,816964648262691053,816964165577979273,816964165573784968,7,NaN
2,1,C6H11O5P,[M + K]+,C6H11KO5P+,Fatty acids,0.831,Fatty Acids and Conjugates,0.710,Halogenated fatty acids,0.028,...,0.633,Organic compounds; Lipids and lipid-like molec...,232.998,37,0.620,816964725454676182,816964165703808398,816964165703808397,26,NaN
3,1,C11H24N2O5,[M + H]+,C11H25N2O5+,Amino acids and Peptides,0.833,Small peptides,0.614,Dipeptides,0.461,...,0.591,Organic compounds; Lipids and lipid-like molec...,265.175,37,0.621,816964648371742993,816964165812860307,816964165812860306,27,NaN
4,1,C7H13O5P,[M + K]+,C7H13KO5P+,Fatty acids,0.888,Fatty Acids and Conjugates,0.719,Halogenated fatty acids,0.014,...,0.525,Organic compounds; Lipids and lipid-like molec...,247.014,37,0.623,816964672669350467,816964166160987554,816964166160987553,40,NaN


In [5]:
# Keep CANOPUS classification columns
canopus_cols_desired = [
    'mappingFeatureId', 'molecularFormula', 'adduct',
    'NPC#pathway', 'NPC#pathway Probability',
    'NPC#superclass', 'NPC#superclass Probability',
    'NPC#class', 'NPC#class Probability',
    'ClassyFire#superclass', 'ClassyFire#superclass probability',
    'ClassyFire#class', 'ClassyFire#class Probability',
    'ClassyFire#subclass', 'ClassyFire#subclass Probability',
    'ClassyFire#level 5', 'ClassyFire#level 5 Probability',
    'ClassyFire#most specific class', 'ClassyFire#most specific class Probability'
]
canopus_cols_keep = [c for c in canopus_cols_desired if c in canopus_annotations.columns]
sirius_canopus = canopus_annotations[canopus_cols_keep].copy()
print(f"CANOPUS selected shape: {sirius_canopus.shape}")
sirius_canopus.head()

CANOPUS selected shape: (984, 19)


,mappingFeatureId,molecularFormula,adduct,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,ClassyFire#superclass,ClassyFire#superclass probability,ClassyFire#class,ClassyFire#class Probability,ClassyFire#subclass,ClassyFire#subclass Probability,ClassyFire#level 5,ClassyFire#level 5 Probability,ClassyFire#most specific class,ClassyFire#most specific class Probability
0,6,C6H6FO3PS,[M + K]+,Fatty acids,0.075,Pseudoalkaloids (transamidation),0.061,Halogenated fatty acids,0.030,Organic acids and derivatives,0.905,Organic phosphonic acids and derivatives,0.948,Organic phosphonic acids,0.815,NaN,NaN,Organic phosphonic acids,0.815
1,7,C16H32N2O10,[M + H]+,Carbohydrates,0.990,Aminosugars and aminoglycosides,0.765,Amino cyclitols,0.730,Organic oxygen compounds,1.000,Organooxygen compounds,1.000,Carbohydrates and carbohydrate conjugates,0.962,Aminosaccharides,0.885,2-deoxystreptamine aminoglycosides,0.528
2,26,C6H11O5P,[M + K]+,Fatty acids,0.831,Fatty Acids and Conjugates,0.710,Halogenated fatty acids,0.028,Lipids and lipid-like molecules,0.725,Fatty Acyls,0.633,NaN,NaN,NaN,NaN,Fatty Acyls,0.633
3,27,C11H24N2O5,[M + H]+,Amino acids and Peptides,0.833,Small peptides,0.614,Dipeptides,0.461,Organic acids and derivatives,0.999,Carboxylic acids and derivatives,0.997,"Amino acids, peptides, and analogues",0.721,Amino acids and derivatives,0.717,Alpha amino acids and derivatives,0.591
4,40,C7H13O5P,[M + K]+,Fatty acids,0.888,Fatty Acids and Conjugates,0.719,Halogenated fatty acids,0.014,Lipids and lipid-like molecules,0.525,NaN,NaN,NaN,NaN,NaN,NaN,Lipids and lipid-like molecules,0.525


### 1.3 Merge SIRIUS Structure + CANOPUS

In [6]:
# Merge CANOPUS (all features with formula) with SIRIUS structure (left join)
sirius_annotations = pd.merge(
    sirius_canopus,
    sirius_struct_clean,
    on='mappingFeatureId',
    how='left'
)

# Add sirius_ prefix to all columns except mappingFeatureId
sirius_annotations = sirius_annotations.rename(
    columns={col: f"sirius_{col}" for col in sirius_annotations.columns if col != 'mappingFeatureId'}
)

print(f"SIRIUS merged annotations shape: {sirius_annotations.shape}")
print(f"Features with SIRIUS structure: {sirius_annotations['sirius_InChIkey2D'].notna().sum()}")
print(f"High-confidence SIRIUS: {(sirius_annotations['sirius_high_confidence_sirius'] == True).sum()}")
sirius_annotations.head()

SIRIUS merged annotations shape: (984, 27)
Features with SIRIUS structure: 984
High-confidence SIRIUS: 125


,mappingFeatureId,sirius_molecularFormula,sirius_adduct,sirius_NPC#pathway,sirius_NPC#pathway Probability,sirius_NPC#superclass,sirius_NPC#superclass Probability,sirius_NPC#class,sirius_NPC#class Probability,sirius_ClassyFire#superclass,...,sirius_ClassyFire#most specific class,sirius_ClassyFire#most specific class Probability,sirius_ConfidenceScoreApproximate,sirius_InChIkey2D,sirius_InChI,sirius_name,sirius_smiles,sirius_dbflags,sirius_links,sirius_high_confidence_sirius
0,6,C6H6FO3PS,[M + K]+,Fatty acids,0.075,Pseudoalkaloids (transamidation),0.061,Halogenated fatty acids,0.030,Organic acids and derivatives,...,Organic phosphonic acids,0.815,0.125,RDAVYJKCJHOMSD,"InChI=1S/C6H6FO3PS/c7-4-1-2-6(12)5(3-4)11(8,9)...",5-Fluoro-2-sulfanyl-phenylphosphonic acid,C1=CC(=C(C=C1F)P(=O)(O)O)S,16777282,PUBCHEMANNOTATIONBIO:(null);PUBCHEM:(49799138)...,False
1,7,C16H32N2O10,[M + H]+,Carbohydrates,0.990,Aminosugars and aminoglycosides,0.765,Amino cyclitols,0.730,Organic oxygen compounds,...,2-deoxystreptamine aminoglycosides,0.528,0.180,TVBWHGVMDKQHJJ,InChI=1S/C16H32N2O10/c19-5-7-9(21)11(23)13(25)...,"N,N'-Bis(alpha-D-mannopyranosyl)-1,4-butanedia...",C(CCNC1C(C(C(C(O1)CO)O)O)O)CNC2C(C(C(C(O2)CO)O...,2,PUBCHEM:(101202401),False
2,26,C6H11O5P,[M + K]+,Fatty acids,0.831,Fatty Acids and Conjugates,0.710,Halogenated fatty acids,0.028,Lipids and lipid-like molecules,...,Fatty Acyls,0.633,0.157,HRTGCDRCJQKACR,"InChI=1S/C6H11O5P/c1-5(6(7)9-2)12(8,10-3)11-4/...",Methyl 2-(dimethoxyphosphinyl)acrylate,COC(=O)C(=C)P(=O)(OC)OC,137506324482,NORMAN:(NS00033255);PUBCHEMANNOTATIONSAFETYAND...,False
3,27,C11H24N2O5,[M + H]+,Amino acids and Peptides,0.833,Small peptides,0.614,Dipeptides,0.461,Organic acids and derivatives,...,Alpha amino acids and derivatives,0.591,0.007,BIOQXNMMINDLQA,InChI=1S/C11H24N2O5/c1-15-4-5-16-6-7-17-8-9-18...,N-(2-aminoethyl)-2-[2-[2-(2-methoxyethoxy)etho...,COCCOCCOCCOCC(=O)NCCN,2,PUBCHEM:(113426778),False
4,40,C7H13O5P,[M + K]+,Fatty acids,0.888,Fatty Acids and Conjugates,0.719,Halogenated fatty acids,0.014,Lipids and lipid-like molecules,...,Lipids and lipid-like molecules,0.525,0.235,STCBUTTVBMCYJL,"InChI=1S/C7H13O5P/c1-13(12,4-2-6(8)9)5-3-7(10)...","Propanoic acid, 3,3'-(methylphosphinylidene)bis-",CP(=O)(CCC(=O)O)CCC(=O)O,137439215618,NORMAN:(NS00057775);PUBCHEM:(20707859 103972);...,False


### 1.4 Load GNPS Annotations

In [7]:
gnps_raw = pd.read_csv(f"{DATA_PATH}/merged_results_with_gnps.tsv", sep="\t")
print(f"GNPS raw shape: {gnps_raw.shape}")

# Select columns (guard against missing ones across different GNPS exports)
gnps_cols_desired = [
    '#Scan#', 'Compound_Name', 'Smiles', 'INCHI', 'InChIKey-Planar',
    'superclass', 'class', 'subclass',
    'npclassifier_superclass', 'npclassifier_class', 'npclassifier_pathway'
]
gnps_cols_keep = [c for c in gnps_cols_desired if c in gnps_raw.columns]
gnps_annotations = gnps_raw[gnps_cols_keep].copy()

# Rename columns with gnps_ prefix
gnps_rename = {
    '#Scan#': 'mappingFeatureId',
    'Compound_Name': 'gnps_Compound_Name',
    'Smiles': 'gnps_smiles',
    'INCHI': 'gnps_InChI',
    'InChIKey-Planar': 'gnps_InChIkey2D',
    'superclass': 'gnps_ClassyFire#superclass',
    'class': 'gnps_ClassyFire#class',
    'subclass': 'gnps_ClassyFire#subclass',
    'npclassifier_superclass': 'gnps_NPC#superclass',
    'npclassifier_class': 'gnps_NPC#class',
    'npclassifier_pathway': 'gnps_NPC#pathway'
}
gnps_annotations = gnps_annotations.rename(columns=gnps_rename)

print(f"GNPS cleaned shape: {gnps_annotations.shape}")
print("Columns:", gnps_annotations.columns.tolist())
gnps_annotations.head()

GNPS raw shape: (66, 46)
GNPS cleaned shape: (66, 11)
Columns: ['mappingFeatureId', 'gnps_Compound_Name', 'gnps_smiles', 'gnps_InChI', 'gnps_InChIkey2D', 'gnps_ClassyFire#superclass', 'gnps_ClassyFire#class', 'gnps_ClassyFire#subclass', 'gnps_NPC#superclass', 'gnps_NPC#class', 'gnps_NPC#pathway']


,mappingFeatureId,gnps_Compound_Name,gnps_smiles,gnps_InChI,gnps_InChIkey2D,gnps_ClassyFire#superclass,gnps_ClassyFire#class,gnps_ClassyFire#subclass,gnps_NPC#superclass,gnps_NPC#class,gnps_NPC#pathway
0,4349,GalCer(d18:2/18:1); [M+H]+ C42H78N1O8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3978,PC(0:0/16:0); [M+H]+ C24H51N1O7P1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3336,PE(16:0/0:0); [M+H]+ C21H45N1O7P1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3329,Spectral Match to 1-Palmitoyl-2-hydroxy-sn-gly...,CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)(O)OCCN)O,InChI=1S/C21H44NO7P/c1-2-3-4-5-6-7-8-9-10-11-1...,YVYMBNSKXOXSKW,Lipids and lipid-like molecules,Glycerophospholipids,Glycerophosphoethanolamines,Glycerophospholipids,Glycerophosphoethanolamines,Fatty acids
4,4114,PC(0:0/18:1); [M+H]+ C26H53N1O7P1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.5 Load GNPS Suspect Annotations

In [8]:
gnps_suspect_raw = pd.read_csv(
    f"{DATA_PATH}/merged_results_with_gnps_suspect.tsv", sep="\t"
)
gnps_suspect_raw = gnps_suspect_raw[
    gnps_suspect_raw["LibraryName"] == "GNPS-SUSPECTLIST.mgf"
]

# Exclude features already in the main GNPS library
gnps_library_ids = gnps_annotations["mappingFeatureId"].tolist()
gnps_suspect_raw = gnps_suspect_raw[
    ~gnps_suspect_raw["#Scan#"].isin(gnps_library_ids)
]

gnps_suspect_annotations = (
    gnps_suspect_raw[['#Scan#', 'Compound_Name']]
    .rename(columns={'#Scan#': 'mappingFeatureId', 'Compound_Name': 'gnps_name'})
)
print(f"GNPS suspect (after filtering): {gnps_suspect_annotations.shape}")
gnps_suspect_annotations.head()

GNPS suspect (after filtering): (70, 2)


,mappingFeatureId,gnps_name
11,3287,Suspect related to 1-Oleoyl-sn-glycero-3-phosp...
14,3946,Suspect related to Spectral Match to 1-Myristo...
16,3017,Suspect related to 1-Oleoyl-sn-glycero-3-phosp...
21,3305,Suspect related to Lithocholic acid (predicted...
23,3782,Suspect related to Spectral Match to 1-(9Z-Oct...


In [9]:
def parse_suspect_annotation(text):
    """Parse GNPS suspect annotation string into structured fields."""
    if pd.isna(text) or not isinstance(text, str):
        return {'compound_name': None, 'sirius_formula': None, 'buddy_formula': None,
                'delta_mz': None, 'explanation': None, 'adduct': None}
    result = {}
    m = re.search(r'Suspect related to\s+(?:Spectral Match to\s+)?(.+?)\s+(?:from [A-Z0-9]+\s+)?\(predicted', text)
    result['compound_name'] = m.group(1).strip() if m else None
    for key, pattern in [
        ('sirius_formula', r'SIRIUS:\s*([A-Z][A-Za-z0-9]+)'),
        ('buddy_formula',  r'BUDDY:\s*([A-Z][A-Za-z0-9]+)'),
        ('explanation',    r'putative explanation:\s*([^;]+)'),
        ('adduct',         r'\[([^\]]+)\](?!.*\[)'),
    ]:
        m2 = re.search(pattern, text)
        result[key] = m2.group(1).strip() if m2 else None
    m3 = re.search(r'delta m/z\s+([\-\d.]+)', text)
    result['delta_mz'] = float(m3.group(1)) if m3 else None
    return result

parsed_df = pd.DataFrame(
    gnps_suspect_annotations['gnps_name'].apply(parse_suspect_annotation).tolist(),
    index=gnps_suspect_annotations.index
)
suspect_final = pd.concat([gnps_suspect_annotations, parsed_df], axis=1).rename(columns={
    'compound_name': 'gnps_suspect_compound_name',
    'sirius_formula': 'gnps_suspect_sirius_formula',
    'buddy_formula':  'gnps_suspect_buddy_formula',
    'delta_mz':       'gnps_suspect_delta_mz',
    'explanation':    'gnps_suspect_explanation',
    'adduct':         'gnps_suspect_adduct',
})
print(f"Suspect final shape: {suspect_final.shape}")
suspect_final.head()

Suspect final shape: (70, 8)


,mappingFeatureId,gnps_name,gnps_suspect_compound_name,gnps_suspect_sirius_formula,gnps_suspect_buddy_formula,gnps_suspect_explanation,gnps_suspect_adduct,gnps_suspect_delta_mz
11,3287,Suspect related to 1-Oleoyl-sn-glycero-3-phosp...,1-Oleoyl-sn-glycero-3-phosphoethanolamine,C22H44NO7P,C22H44NO7P,Ala->Gly substitution|Gln->Asn substitution|Gl...,M+Na,-14.016
14,3946,Suspect related to Spectral Match to 1-Myristo...,1-Myristoyl-sn-glycero-3-phosphocholine,C21H49N6O5P,C28H44N6O2,unspecified,M+Na,29.049
16,3017,Suspect related to 1-Oleoyl-sn-glycero-3-phosp...,1-Oleoyl-sn-glycero-3-phosphoethanolamine,C28H37NO4,C21H42NO7P,"Desethylation, didemethylation, dealkylation|M...",M+Na,-28.031
21,3305,Suspect related to Lithocholic acid (predicted...,Lithocholic acid,C29H50O3,C29H50O3,Labeling transglutaminase substrate on glutami...,M-H2O+H,70.078
23,3782,Suspect related to Spectral Match to 1-(9Z-Oct...,1-(9Z-Octadecenoyl)-sn-glycero-3-phosphocholine,C16H47N12O4P,C31H53NO5,Cys->Thr substitution,M+H,-1.959


### 1.6 Build Feature Base from `cocoyam_b3_iimn_fbmn_quant.csv`

In [10]:
ms2_df = pd.read_csv(f"{DATA_PATH}/cocoyam_b3_iimn_fbmn_quant.csv")
# 'row ID' is the GNPS/SIRIUS feature identifier
feature_base = ms2_df.rename(columns={'row ID': 'mappingFeatureId', 'row m/z': 'mz', 'row retention time': 'rt'})

print(f"Feature base shape: {feature_base.shape}")
feature_base[["mappingFeatureId","mz","rt"]].head()

Feature base shape: (1840, 20)


,mappingFeatureId,mz,rt
0,4,308.216567,0.561364
1,6,246.937392,0.593106
2,7,413.211358,0.605967
3,26,232.998154,0.620453
4,27,265.174561,0.620717


### 1.7 Merge All Annotations

In [11]:
# Build combined df: feature base → left join SIRIUS → left join GNPS → left join Suspect
df_combined = (
    feature_base
    .merge(sirius_annotations, on='mappingFeatureId', how='left')
    .merge(gnps_annotations,   on='mappingFeatureId', how='left')
    .merge(suspect_final,       on='mappingFeatureId', how='left')
)

print(f"Combined df shape: {df_combined.shape}")
print(f"  Total features:             {len(df_combined)}")
print(f"  With GNPS annotations:      {df_combined['gnps_Compound_Name'].notna().sum()}")
print(f"  With SIRIUS structure:      {df_combined['sirius_InChIkey2D'].notna().sum()}")
print(f"  With SIRIUS CANOPUS:        {df_combined['sirius_molecularFormula'].notna().sum()}")
print(f"  With Suspect annotations:   {df_combined['gnps_suspect_compound_name'].notna().sum()}")
df_combined.head()

Combined df shape: (1840, 63)
  Total features:             1840
  With GNPS annotations:      66
  With SIRIUS structure:      984
  With SIRIUS CANOPUS:        984
  With Suspect annotations:   60


,mappingFeatureId,mz,rt,row ion mobility,row ion mobility unit,row CCS,correlation group ID,annotation network number,best ion,auto MS2 verify,...,gnps_NPC#superclass,gnps_NPC#class,gnps_NPC#pathway,gnps_name,gnps_suspect_compound_name,gnps_suspect_sirius_formula,gnps_suspect_buddy_formula,gnps_suspect_explanation,gnps_suspect_adduct,gnps_suspect_delta_mz
0,4,308.216567,0.561364,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6,246.937392,0.593106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,413.211358,0.605967,NaN,NaN,NaN,7.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,26,232.998154,0.620453,NaN,NaN,NaN,7.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,27,265.174561,0.620717,NaN,NaN,NaN,7.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.8 Assign Annotation Source

In [12]:
has_gnps    = df_combined['gnps_Compound_Name'].notna()
has_sirius  = (
    df_combined['sirius_InChIkey2D'].notna() &
    (df_combined['sirius_high_confidence_sirius'] == True)
)
has_suspect = df_combined['gnps_suspect_compound_name'].notna()

df_combined['annotation_source'] = 'Unannotated'

for idx in df_combined.index:
    sources = []
    if has_gnps.loc[idx]:    sources.append('gnps')
    if has_sirius.loc[idx]:  sources.append('sirius')
    if has_suspect.loc[idx]: sources.append('suspect')
    if sources:
        df_combined.loc[idx, 'annotation_source'] = ':'.join(sources)

# InChIKey cross-match where both sources annotated the same feature
both_with_inchikeys = (
    df_combined['gnps_InChIkey2D'].notna() &
    df_combined['sirius_InChIkey2D'].notna()
)
df_combined['inchikey_match'] = None
df_combined.loc[both_with_inchikeys, 'inchikey_match'] = (
    df_combined.loc[both_with_inchikeys, 'gnps_InChIkey2D'] ==
    df_combined.loc[both_with_inchikeys, 'sirius_InChIkey2D']
)

print("=== Annotation Source Counts ===")
print(df_combined['annotation_source'].value_counts())
print(f"\nInChIKey matches (GNPS == SIRIUS): {df_combined['inchikey_match'].sum()}")

=== Annotation Source Counts ===
annotation_source
Unannotated       1615
sirius              99
gnps                52
suspect             48
gnps:sirius         14
sirius:suspect      12
Name: count, dtype: int64

InChIKey matches (GNPS == SIRIUS): 6


### 1.9 Save MS2 Annotations

In [13]:
# Build final column list — keep only columns that exist
final_cols_desired = [
    'mappingFeatureId', 'mz', 'rt',
    'correlation group ID', 'best ion', 'auto MS2 verify',
    'identified by n=', 'partners', 'neutral M mass',
    # GNPS
    'gnps_Compound_Name', 'gnps_smiles', 'gnps_InChI', 'gnps_InChIkey2D',
    'gnps_ClassyFire#superclass', 'gnps_ClassyFire#class', 'gnps_ClassyFire#subclass',
    'gnps_NPC#superclass', 'gnps_NPC#class', 'gnps_NPC#pathway',
    # SIRIUS
    'sirius_molecularFormula', 'sirius_adduct',
    'sirius_NPC#pathway', 'sirius_NPC#pathway Probability',
    'sirius_NPC#superclass', 'sirius_NPC#superclass Probability',
    'sirius_NPC#class', 'sirius_NPC#class Probability',
    'sirius_ClassyFire#superclass', 'sirius_ClassyFire#superclass probability',
    'sirius_ClassyFire#class', 'sirius_ClassyFire#class Probability',
    'sirius_ClassyFire#subclass', 'sirius_ClassyFire#subclass Probability',
    'sirius_ClassyFire#level 5', 'sirius_ClassyFire#level 5 Probability',
    'sirius_ClassyFire#most specific class', 'sirius_ClassyFire#most specific class Probability',
    'sirius_ConfidenceScoreApproximate', 'sirius_name', 'sirius_InChIkey2D',
    'sirius_InChI', 'sirius_smiles', 'sirius_dbflags', 'sirius_links',
    'sirius_high_confidence_sirius',
    # Suspect
    'gnps_suspect_compound_name', 'gnps_suspect_sirius_formula',
    'gnps_suspect_buddy_formula', 'gnps_suspect_delta_mz',
    'gnps_suspect_explanation', 'gnps_suspect_adduct',
    # Metadata
    'annotation_source', 'inchikey_match'
]
final_cols = [c for c in final_cols_desired if c in df_combined.columns]
final_ms2_annotations = df_combined[final_cols].copy()

output_path = f"../results/{BATCH_ID}_ms2_annotations.csv"
final_ms2_annotations.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Total features:  {len(final_ms2_annotations)}")
print(f"Annotated:       {(final_ms2_annotations['annotation_source'] != 'Unannotated').sum()}")
print(f"Unannotated:     {(final_ms2_annotations['annotation_source'] == 'Unannotated').sum()}")
final_ms2_annotations.head()

Saved: ../results/b3_cocoyamonly_ms2_annotations.csv
Total features:  1840
Annotated:       225
Unannotated:     1615


,mappingFeatureId,mz,rt,correlation group ID,best ion,auto MS2 verify,identified by n=,partners,neutral M mass,gnps_Compound_Name,...,sirius_links,sirius_high_confidence_sirius,gnps_suspect_compound_name,gnps_suspect_sirius_formula,gnps_suspect_buddy_formula,gnps_suspect_delta_mz,gnps_suspect_explanation,gnps_suspect_adduct,annotation_source,inchikey_match
0,4,308.216567,0.561364,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
1,6,246.937392,0.593106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,PUBCHEMANNOTATIONBIO:(null);PUBCHEM:(49799138)...,False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
2,7,413.211358,0.605967,7.0,NaN,NaN,NaN,NaN,NaN,NaN,...,PUBCHEM:(101202401),False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
3,26,232.998154,0.620453,7.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NORMAN:(NS00033255);PUBCHEMANNOTATIONSAFETYAND...,False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None
4,27,265.174561,0.620717,7.0,NaN,NaN,NaN,NaN,NaN,NaN,...,PUBCHEM:(113426778),False,NaN,NaN,NaN,NaN,NaN,NaN,Unannotated,None


---
## Section 2 — Quantitative Analysis (CYF only)

Peak areas are normalized by sample weight (1.50 g control; 2.64, 2.06, 2.06 g fermented replicates), filtered for prevalence (detected in ≥ 2 of 3 replicates in at least one group), and missing values are imputed with uniform random noise below the second-smallest observed intensity. After log2 transformation, Welch t-tests compare control vs. fermented, with Benjamini-Hochberg FDR correction. Raw p-values are also saved separately for mummichog pathway analysis.

In [14]:
quant_data = pd.read_csv(f"{DATA_PATH}/b3_full_feature_table_cocoyam.csv")
print(f"Feature table shape: {quant_data.shape}")
quant_data.head()

Feature table shape: (4468, 154)


,id,mz,mz_range:min,mz_range:max,rt,rt_range:min,rt_range:max,area,height,intensity_range:min,...,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:area,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:height,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:intensity_range:min,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:intensity_range:max,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:charge,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:fragment_scans,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:isotopes,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:tailing_factor,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:asymmetry_factor,datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:rt_ms2_apex_distance
0,1,164.98286,164.98202,164.98356,0.5400,0.5013,0.5955,229.0,4238.0,755.3,...,218.50,3351.0,755.3,3351.0,1.0,NaN,2.0,1.9207,2.8413,NaN
1,2,132.95725,132.95675,132.95772,0.5528,0.5011,0.5883,290.1,4607.0,1229.0,...,88.86,4607.0,2519.0,4607.0,NaN,NaN,NaN,0.7006,0.4011,NaN
2,3,470.26896,470.25600,470.27142,0.5575,0.5344,0.6042,108.8,2971.0,747.2,...,104.70,2760.0,747.2,2760.0,NaN,NaN,NaN,1.0240,1.0480,NaN
3,4,308.21657,308.21538,308.22046,0.5614,0.5309,0.6600,413.1,6106.0,749.2,...,413.10,5635.0,904.0,5635.0,1.0,1.0,2.0,2.4414,3.9066,-0.0123
4,5,146.16417,146.14775,146.16506,0.6435,0.5379,0.6780,332.9,5508.0,751.6,...,332.90,5124.0,830.4,5124.0,NaN,NaN,NaN,0.6312,0.2535,NaN


In [15]:
area_mapping = {
    'datafile:Mots_18102024_Foodomics_CYF1_S52.mzML:area': 'CYF_1_rep_1',
    'datafile:Mots_18102024_Foodomics_CYF1_S53.mzML:area': 'CYF_1_rep_2',
    'datafile:Mots_18102024_Foodomics_CYF1_S54.mzML:area': 'CYF_1_rep_3',
    'datafile:Mots_18102024_Foodomics_CYF2_S55.mzML:area': 'CYF_2_rep_1',
    'datafile:Mots_18102024_Foodomics_CYF2_S56.mzML:area': 'CYF_2_rep_2',
    'datafile:Mots_18102024_Foodomics_CYF2_S57.mzML:area': 'CYF_2_rep_3',
}

feature_list_renamed = quant_data.rename(columns=area_mapping)
feature_list_clean = feature_list_renamed[['id', 'mz', 'rt'] + list(area_mapping.values())].copy()

print(f"Feature list shape: {feature_list_clean.shape}")
feature_list_clean.head()

Feature list shape: (4468, 9)


,id,mz,rt,CYF_1_rep_1,CYF_1_rep_2,CYF_1_rep_3,CYF_2_rep_1,CYF_2_rep_2,CYF_2_rep_3
0,1,164.98286,0.5400,229.00,105.00,65.38,180.7,52.74,218.50
1,2,132.95725,0.5528,99.02,73.12,281.30,290.1,63.24,88.86
2,3,470.26896,0.5575,35.81,43.85,39.06,108.8,104.60,104.70
3,4,308.21657,0.5614,110.60,229.00,233.90,405.7,242.80,413.10
4,5,146.16417,0.6435,247.40,167.30,177.60,328.1,215.20,332.90


In [16]:
SAMPLE_WEIGHTS = {
    'CYF_1_rep_1': 1.50,  # Cocoyam Control
    'CYF_1_rep_2': 1.50,
    'CYF_1_rep_3': 1.50,
    'CYF_2_rep_1': 2.64,  # Cocoyam Fermented
    'CYF_2_rep_2': 2.06,
    'CYF_2_rep_3': 2.06,
}

for sample, weight in SAMPLE_WEIGHTS.items():
    feature_list_clean[f"{sample}_norm"] = feature_list_clean[sample] / weight

print("Normalization complete.")
print(f"Total columns: {len(feature_list_clean.columns)}")

Normalization complete.
Total columns: 15


In [17]:
CYF1_cols = [c for c in feature_list_clean.columns if c.startswith('CYF_1') and c.endswith('_norm')]
CYF2_cols = [c for c in feature_list_clean.columns if c.startswith('CYF_2') and c.endswith('_norm')]
all_norm_cols = CYF1_cols + CYF2_cols

print(f"CYF_1 norm cols: {CYF1_cols}")
print(f"CYF_2 norm cols: {CYF2_cols}")

# ── Global prevalence filter: >=2 non-zero in at least one group ────────────
def global_prevalence_filter(df, min_present=2):
    cyf1_ok = df[CYF1_cols].notna().sum(axis=1) >= min_present
    cyf2_ok = df[CYF2_cols].notna().sum(axis=1) >= min_present
    return df[cyf1_ok | cyf2_ok].copy()

n_before = len(feature_list_clean)
feature_list_filtered = global_prevalence_filter(feature_list_clean, min_present=2)
n_after = len(feature_list_filtered)
print(f"\nPrevalence filter (>=2/3 reps in at least one group):")
print(f"  Before: {n_before:,}")
print(f"  After:  {n_after:,}")
print(f"  Removed: {n_before - n_after:,} ({100*(n_before-n_after)/n_before:.1f}%)")

CYF_1 norm cols: ['CYF_1_rep_1_norm', 'CYF_1_rep_2_norm', 'CYF_1_rep_3_norm']
CYF_2 norm cols: ['CYF_2_rep_1_norm', 'CYF_2_rep_2_norm', 'CYF_2_rep_3_norm']

Prevalence filter (>=2/3 reps in at least one group):
  Before: 4,468
  After:  4,437
  Removed: 31 (0.7%)


In [18]:
np.random.seed(RANDOM_SEED)
feature_list_imputed = feature_list_filtered.copy()

flat_vals = feature_list_filtered[all_norm_cols].to_numpy(dtype=float).ravel()
positive_vals = flat_vals[np.isfinite(flat_vals) & (flat_vals > 0)]
unique_pos = np.unique(positive_vals)
second_min_positive = unique_pos[1] if unique_pos.size > 1 else unique_pos[0]

n_imputed_total = 0
for col in all_norm_cols:
    missing_mask = feature_list_imputed[col].isna() | (feature_list_imputed[col] == 0)
    n_missing = missing_mask.sum()
    if n_missing > 0:
        noise = np.random.uniform(0, second_min_positive, size=n_missing)
        noise = np.clip(noise, 1e-10, None)
        feature_list_imputed.loc[missing_mask, col] = noise
        n_imputed_total += n_missing

print(f"Noise imputation (uniform[0, second_min], seed={RANDOM_SEED}):")
print(f"  second_min_positive: {second_min_positive:.3e}")
print(f"  Total values imputed: {n_imputed_total:,}")

# ── log2 transform ────────────────────────────────────────────────────────
for col in all_norm_cols:
    feature_list_imputed[col.replace('_norm', '_norm_log2')] = np.log2(feature_list_imputed[col])

log2_vals = feature_list_imputed[[c.replace('_norm','_norm_log2') for c in all_norm_cols]].values.flatten()
print(f"\nLog2 transform complete:")
print(f"  Min: {log2_vals.min():.2f}, Max: {log2_vals.max():.2f}, Mean: {log2_vals.mean():.2f}")

Noise imputation (uniform[0, second_min], seed=42):
  second_min_positive: 7.708e-01
  Total values imputed: 1,037

Log2 transform complete:
  Min: -8.13, Max: 16.12, Mean: 6.53


In [19]:
def run_comparison(df, group1_name, group2_name, output_filename, fdr=False):
    g1_log2 = [c for c in df.columns if c.startswith(group1_name) and c.endswith('_norm_log2')]
    g2_log2 = [c for c in df.columns if c.startswith(group2_name) and c.endswith('_norm_log2')]

    g1_data = df[g1_log2]
    g2_data = df[g2_log2]

    log2fc = g2_data.mean(axis=1).values - g1_data.mean(axis=1).values
    t_stat, p_value = stats.ttest_ind(g1_data, g2_data, axis=1, equal_var=False)

    results = df[['id', 'mz', 'rt']].copy()
    results['log2FC']  = log2fc
    results['t.score'] = t_stat
    results['p.value'] = p_value
    results = results.dropna(subset=['t.score', 'p.value'])

    if fdr:
        _, p_adj, _, _ = multipletests(results['p.value'], alpha=0.05, method='fdr_bh')
        results['p.value'] = p_adj

    results = results.sort_values('p.value')
    results.to_csv(output_filename, index=False)

    label = f"{group1_name} vs {group2_name}"
    fdr_label = " [FDR-BH]" if fdr else ""
    print(f"  {label}{fdr_label}")
    print(f"    Features tested:        {len(results):,}")
    print(f"    p < 0.05:               {(results['p.value'] < 0.05).sum():,}")
    print(f"    |log2FC|>1 & p<0.05:    {((abs(results['log2FC'])>1) & (results['p.value']<0.05)).sum():,}\n")
    return results

print("="*60)
print("COCOYAM COMPARISONS")
print(f"Features in analysis: {len(feature_list_imputed):,}")
print("="*60)

print("\n--- Raw p-values (mummichog/pathway analysis) ---")
cocoyam_raw = run_comparison(
    feature_list_imputed, 'CYF_1', 'CYF_2',
    '../results/mummichog_cocoyam_vs_fermented_cocoyam.csv', fdr=False
)

print("--- FDR-adjusted p-values (volcano plots) ---")
cocoyam_fdr = run_comparison(
    feature_list_imputed, 'CYF_1', 'CYF_2',
    f'../results/{BATCH_ID}_fdr_cocoyam_vs_fermented_cocoyam.csv', fdr=True
)

feature_list_imputed.to_csv(f'../results/{BATCH_ID}_feature_list_imputed_log2.csv', index=False)
print(f"Imputed feature table saved: {BATCH_ID}_feature_list_imputed_log2.csv")

COCOYAM COMPARISONS
Features in analysis: 4,437

--- Raw p-values (mummichog/pathway analysis) ---
  CYF_1 vs CYF_2
    Features tested:        4,437
    p < 0.05:               2,258
    |log2FC|>1 & p<0.05:    1,443

--- FDR-adjusted p-values (volcano plots) ---
  CYF_1 vs CYF_2 [FDR-BH]
    Features tested:        4,437
    p < 0.05:               1,292
    |log2FC|>1 & p<0.05:    1,013

Imputed feature table saved: b3_cocoyamonly_feature_list_imputed_log2.csv


---
## Section 3 — BioTransformer Prep (InChIKey → SMILES)

InChI strings from high-confidence SIRIUS and GNPS annotations are converted to canonical SMILES using RDKit, then standardized (fragment parent selection, charge neutralization, tautomer canonicalization). The output CSV feeds into `biotransformer_sirius.py` for metabolic transformation prediction with BioTransformer 3.0.

In [20]:
annotations = pd.read_csv(f'../results/{BATCH_ID}_ms2_annotations.csv')
print(f"Total features: {len(annotations)}")
print("Annotation source breakdown:")
print(annotations['annotation_source'].value_counts())

# Keep only annotated features
annotations = annotations[annotations['annotation_source'] != 'Unannotated'].copy()

# For GNPS-only rows, clear SIRIUS InChIKey/name to avoid confusion
gnps_only_mask = annotations['annotation_source'] == 'gnps'
annotations.loc[gnps_only_mask, 'sirius_InChIkey2D'] = np.nan
annotations.loc[gnps_only_mask, 'sirius_name']       = np.nan

print(f"\nAnnotated features for BioTransformer: {len(annotations)}")

Total features: 1840
Annotation source breakdown:
annotation_source
Unannotated       1615
sirius              99
gnps                52
suspect             48
gnps:sirius         14
sirius:suspect      12
Name: count, dtype: int64

Annotated features for BioTransformer: 225


In [21]:
annotations_with_inchi = annotations[
    (annotations['annotation_source'].str.contains('gnps', na=False)) |
    (annotations['annotation_source'].str.contains('sirius', na=False))
][['mappingFeatureId', 'gnps_InChIkey2D', 'sirius_InChIkey2D',
   'gnps_InChI', 'sirius_InChI', 'annotation_source', 'sirius_high_confidence_sirius']].copy()

annotations_with_inchi = annotations_with_inchi[
    annotations_with_inchi['gnps_InChIkey2D'].notna() |
    annotations_with_inchi['sirius_InChIkey2D'].notna()
]

# Build long-form InChIKey list (deduplicated)
inchikey_list = []
for _, row in annotations_with_inchi.iterrows():
    if pd.notna(row['gnps_InChIkey2D']) and pd.notna(row['gnps_InChI']):
        inchikey_list.append({'inchikey': row['gnps_InChIkey2D'], 'inchi': row['gnps_InChI']})
    if pd.notna(row['sirius_InChIkey2D']) and pd.notna(row['sirius_InChI']):
        inchikey_list.append({'inchikey': row['sirius_InChIkey2D'], 'inchi': row['sirius_InChI']})

inchikey_df = pd.DataFrame(inchikey_list).drop_duplicates(subset=['inchikey'])
only_inchis = inchikey_df[['inchikey', 'inchi']].reset_index(drop=True)

print(f"Unique InChIKeys for BioTransformer: {len(only_inchis)}")
only_inchis.head()

Unique InChIKeys for BioTransformer: 128


,inchikey,inchi
0,LXNHXLLTXMVWPM,InChI=1S/C8H11NO3/c1-5-8(12)7(4-11)6(3-10)2-9-...
1,UYTPUPDQBNUYGX,InChI=1S/C5H5N5O/c6-5-9-3-2(4(11)10-5)7-1-8-3/...
2,FDGQSTZJBFJUBT,InChI=1S/C5H4N4O/c10-5-3-4(7-1-6-3)8-2-9-5/h1-...
3,OUYCCCASQSFEME,InChI=1S/C9H11NO3/c10-8(9(12)13)5-6-1-3-7(11)4...
4,URCADIGEIOYLRQ,InChI=1S/C14H21NO6/c15-6-5-8-1-3-9(4-2-8)20-14...


In [22]:
from rdkit import Chem
from rdkit.Chem import AllChem

def inchi_to_canonical_smiles(inchi):
    """Convert InChI to canonical SMILES."""
    try:
        mol = Chem.MolFromInchi(inchi)
        if mol is not None:
            return Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
    except:
        pass
    return None

def clean_and_convert_inchi(inchi):
    """Clean InChI string and convert to SMILES."""
    if pd.isna(inchi):
        return None
    inchi = str(inchi).strip().strip('"')
    if '-' in inchi and len(inchi) == 27:  # it's an InChIKey, not InChI
        return None
    if not inchi.startswith('InChI='):
        inchi = 'InChI=' + inchi
    return inchi_to_canonical_smiles(inchi)

only_inchis['smiles'] = only_inchis['inchi'].apply(inchi_to_canonical_smiles)
failed = only_inchis['smiles'].isna()
if failed.any():
    only_inchis.loc[failed, 'smiles'] = only_inchis.loc[failed, 'inchi'].apply(clean_and_convert_inchi)

print(f"Total:    {len(only_inchis)}")
print(f"Success:  {only_inchis['smiles'].notna().sum()}")
print(f"Failed:   {only_inchis['smiles'].isna().sum()}")
only_inchis.head()

Total:    128
Success:  127
Failed:   1


[11:37:29] ERROR: 

[11:37:29] ERROR: 

[11:37:29] ERROR: 

[11:37:29] ERROR: 

[11:37:29] ERROR: 

[11:37:29] ERROR: 



,inchikey,inchi,smiles
0,LXNHXLLTXMVWPM,InChI=1S/C8H11NO3/c1-5-8(12)7(4-11)6(3-10)2-9-...,Cc1ncc(CO)c(CO)c1O
1,UYTPUPDQBNUYGX,InChI=1S/C5H5N5O/c6-5-9-3-2(4(11)10-5)7-1-8-3/...,N=c1nc(O)c2nc[nH]c2[nH]1
2,FDGQSTZJBFJUBT,InChI=1S/C5H4N4O/c10-5-3-4(7-1-6-3)8-2-9-5/h1-...,Oc1ncnc2[nH]cnc12
3,OUYCCCASQSFEME,InChI=1S/C9H11NO3/c10-8(9(12)13)5-6-1-3-7(11)4...,NC(Cc1ccc(O)cc1)C(=O)O
4,URCADIGEIOYLRQ,InChI=1S/C14H21NO6/c15-6-5-8-1-3-9(4-2-8)20-14...,NCCc1ccc(OC2OC(CO)C(O)C(O)C2O)cc1


In [23]:
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize(smiles):
    """Standardize SMILES: cleanup, fragment parent, neutralize, canonicalize tautomer."""
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        clean_mol  = rdMolStandardize.Cleanup(mol)
        parent_mol = rdMolStandardize.FragmentParent(clean_mol)
        uncharged  = rdMolStandardize.Uncharger().uncharge(parent_mol)
        tautomer   = rdMolStandardize.TautomerEnumerator().Canonicalize(uncharged)
        return Chem.MolToSmiles(tautomer, canonical=True)
    except:
        return smiles  # fallback: return original

only_inchis['clean_smiles'] = only_inchis['smiles'].apply(standardize)
print(f"Standardized: {only_inchis['clean_smiles'].notna().sum()}/{len(only_inchis)}")
only_inchis.head()

[11:37:29] Initializing MetalDisconnector
[11:37:29] Running MetalDisconnector
[11:37:29] Initializing Normalizer
[11:37:29] Running Normalizer
[11:37:29] Initializing MetalDisconnector
[11:37:29] Running MetalDisconnector
[11:37:29] Initializing Normalizer
[11:37:29] Running Normalizer
[11:37:29] Running LargestFragmentChooser
[11:37:29] Running Uncharger
[11:37:29] Initializing MetalDisconnector
[11:37:29] Running MetalDisconnector
[11:37:29] Initializing Normalizer
[11:37:29] Running Normalizer
[11:37:29] Initializing MetalDisconnector
[11:37:29] Running MetalDisconnector
[11:37:29] Initializing Normalizer
[11:37:29] Running Normalizer
[11:37:29] Running LargestFragmentChooser
[11:37:29] Running Uncharger
[11:37:29] Initializing MetalDisconnector
[11:37:29] Running MetalDisconnector
[11:37:29] Initializing Normalizer
[11:37:29] Running Normalizer
[11:37:29] Initializing MetalDisconnector
[11:37:29] Running MetalDisconnector
[11:37:29] Initializing Normalizer
[11:37:29] Running Norma

Standardized: 127/128


[11:37:33] Initializing MetalDisconnector
[11:37:33] Running MetalDisconnector
[11:37:33] Initializing Normalizer
[11:37:33] Running Normalizer
[11:37:33] Running LargestFragmentChooser
[11:37:33] Running Uncharger
[11:37:33] Initializing MetalDisconnector
[11:37:33] Running MetalDisconnector
[11:37:33] Initializing Normalizer
[11:37:33] Running Normalizer
[11:37:33] Initializing MetalDisconnector
[11:37:33] Running MetalDisconnector
[11:37:33] Initializing Normalizer
[11:37:33] Running Normalizer
[11:37:33] Running LargestFragmentChooser
[11:37:33] Running Uncharger
[11:37:33] Initializing MetalDisconnector
[11:37:33] Running MetalDisconnector
[11:37:33] Initializing Normalizer
[11:37:33] Running Normalizer
[11:37:33] Initializing MetalDisconnector
[11:37:33] Running MetalDisconnector
[11:37:33] Initializing Normalizer
[11:37:33] Running Normalizer
[11:37:33] Running LargestFragmentChooser
[11:37:33] Running Uncharger
[11:37:33] Initializing MetalDisconnector
[11:37:33] Running MetalD

,inchikey,inchi,smiles,clean_smiles
0,LXNHXLLTXMVWPM,InChI=1S/C8H11NO3/c1-5-8(12)7(4-11)6(3-10)2-9-...,Cc1ncc(CO)c(CO)c1O,Cc1ncc(CO)c(CO)c1O
1,UYTPUPDQBNUYGX,InChI=1S/C5H5N5O/c6-5-9-3-2(4(11)10-5)7-1-8-3/...,N=c1nc(O)c2nc[nH]c2[nH]1,Nc1nc(=O)c2[nH]cnc2[nH]1
2,FDGQSTZJBFJUBT,InChI=1S/C5H4N4O/c10-5-3-4(7-1-6-3)8-2-9-5/h1-...,Oc1ncnc2[nH]cnc12,O=c1[nH]cnc2[nH]cnc12
3,OUYCCCASQSFEME,InChI=1S/C9H11NO3/c10-8(9(12)13)5-6-1-3-7(11)4...,NC(Cc1ccc(O)cc1)C(=O)O,NC(Cc1ccc(O)cc1)C(=O)O
4,URCADIGEIOYLRQ,InChI=1S/C14H21NO6/c15-6-5-8-1-3-9(4-2-8)20-14...,NCCc1ccc(OC2OC(CO)C(O)C(O)C2O)cc1,NCCc1ccc(OC2OC(CO)C(O)C(O)C2O)cc1


In [24]:
# Merge back to get mappingFeatureId, mz, rt, compound_name
gnps_df = annotations[['mappingFeatureId', 'annotation_source', 'mz', 'rt',
                        'gnps_InChIkey2D', 'gnps_Compound_Name']].rename(
    columns={'gnps_InChIkey2D': 'inchikey', 'gnps_Compound_Name': 'compound_name'}
)
sirius_df = annotations[['mappingFeatureId', 'annotation_source', 'mz', 'rt',
                          'sirius_InChIkey2D', 'sirius_name']].rename(
    columns={'sirius_InChIkey2D': 'inchikey', 'sirius_name': 'compound_name'}
)
combined_annotations = pd.concat([gnps_df, sirius_df]).drop_duplicates()

result = only_inchis.merge(combined_annotations, on='inchikey', how='left')
result = result[['mappingFeatureId', 'inchikey', 'inchi', 'smiles', 'clean_smiles',
                 'annotation_source', 'mz', 'rt', 'compound_name']]

print(f"BioTransformer input shape: {result.shape}")
result.head()

BioTransformer input shape: (150, 9)


,mappingFeatureId,inchikey,inchi,smiles,clean_smiles,annotation_source,mz,rt,compound_name
0,282,LXNHXLLTXMVWPM,InChI=1S/C8H11NO3/c1-5-8(12)7(4-11)6(3-10)2-9-...,Cc1ncc(CO)c(CO)c1O,Cc1ncc(CO)c(CO)c1O,sirius,170.080129,0.896105,Pyridoxol
1,286,UYTPUPDQBNUYGX,InChI=1S/C5H5N5O/c6-5-9-3-2(4(11)10-5)7-1-8-3/...,N=c1nc(O)c2nc[nH]c2[nH]1,Nc1nc(=O)c2[nH]cnc2[nH]1,sirius,152.055603,0.904497,Guanine
2,314,FDGQSTZJBFJUBT,InChI=1S/C5H4N4O/c10-5-3-4(7-1-6-3)8-2-9-5/h1-...,Oc1ncnc2[nH]cnc12,O=c1[nH]cnc2[nH]cnc12,sirius,137.044780,0.960461,Sarkin
3,315,OUYCCCASQSFEME,InChI=1S/C9H11NO3/c10-8(9(12)13)5-6-1-3-7(11)4...,NC(Cc1ccc(O)cc1)C(=O)O,NC(Cc1ccc(O)cc1)C(=O)O,sirius,182.080114,0.961044,L-Tyr
4,317,URCADIGEIOYLRQ,InChI=1S/C14H21NO6/c15-6-5-8-1-3-9(4-2-8)20-14...,NCCc1ccc(OC2OC(CO)C(O)C(O)C2O)cc1,NCCc1ccc(OC2OC(CO)C(O)C(O)C2O)cc1,sirius,300.142946,0.970792,Tyrosamine beta-D-glucoside


In [ ]:
# Save clean InChI/SMILES reference table
only_inchis.to_csv(f'../results/{BATCH_ID}_clean_inchis_smiles.csv', index=False)
print(f"Saved: ../results/{BATCH_ID}_clean_inchis_smiles.csv")

# Save BioTransformer input file
bt_dir = '../data/biotransformer'
os.makedirs(bt_dir, exist_ok=True)
output_file = os.path.join(bt_dir, f'{BATCH_ID}_compounds_for_biotransformer.csv')
result.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
print(f"  Features: {len(result)}")
print(f"  Columns:  {list(result.columns)}")
print("\n✓ Ready to run: python scripts/biotransformer_sirius.py --batch b3_cocoyamonly")
print('\n If _biot files were run then please ignore the aforementioned step and just continue with figure generation')

Saved: ../results/b3_cocoyamonly_clean_inchis_smiles.csv
Saved: ../data/biotransformer/b3_cocoyamonly_compounds_for_biotransformer.csv
  Features: 150
  Columns:  ['mappingFeatureId', 'inchikey', 'inchi', 'smiles', 'clean_smiles', 'annotation_source', 'mz', 'rt', 'compound_name']

✓ Ready to run: python scripts/biotransformer_sirius.py --batch b3_cocoyamonly
